# Week 7: Model Explainability & Clinical Interpretation

## CalmSense: Multimodal Stress Detection from Physiological Signals

### Overview
This notebook demonstrates comprehensive model explainability techniques for stress detection:

1. **SHAP (SHapley Additive exPlanations)**: Global and local feature importance
2. **LIME (Local Interpretable Model-agnostic Explanations)**: Instance-level explanations
3. **Attention Visualization**: For transformer-based models
4. **Grad-CAM**: For CNN-based models
5. **Clinical Interpretation**: Physiologically-grounded insights

### Learning Objectives
- Understand different explainability methods and their trade-offs
- Interpret model predictions in clinical context
- Generate publication-quality visualizations
- Build confidence in model predictions through transparency

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# CalmSense imports
from src.explainability import (
    SHAPExplainer,
    LIMEExplainer,
    AttentionVisualizer,
    GradCAMExplainer,
    ClinicalInterpreter,
    NORMAL_RANGES,
    StressLevel
)

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Explainability module loaded successfully!")

## 2. Load Pre-trained Model and Data

In [ ]:
# Create synthetic data for demonstration
np.random.seed(42)

# Feature names (physiological markers)
feature_names = [
    # HRV Time Domain
    'hr_mean', 'hrv_sdnn', 'hrv_rmssd', 'hrv_pnn50', 'hrv_nn50',
    # HRV Frequency Domain
    'hrv_lf_power', 'hrv_hf_power', 'hrv_lf_hf_ratio', 'hrv_vlf_power',
    # EDA
    'eda_mean', 'eda_std', 'scr_count', 'scr_amplitude_mean',
    # Respiration
    'resp_rate', 'resp_depth', 'resp_variability',
    # Other
    'temp_mean', 'temp_variability', 'emg_mean', 'emg_activation_count'
]

n_features = len(feature_names)
n_samples = 200

# Generate synthetic features
X = np.random.randn(n_samples, n_features)

# Make some features correlated with stress
y = np.zeros(n_samples, dtype=int)

# Baseline samples (0)
y[:80] = 0
X[:80, 0] = np.random.normal(70, 10, 80)  # Normal HR
X[:80, 2] = np.random.normal(45, 10, 80)  # Normal RMSSD
X[:80, 9] = np.random.normal(3, 1, 80)    # Normal EDA

# Stress samples (1)
y[80:160] = 1
X[80:160, 0] = np.random.normal(95, 15, 80)   # Elevated HR
X[80:160, 2] = np.random.normal(20, 8, 80)    # Low RMSSD
X[80:160, 9] = np.random.normal(8, 2, 80)     # High EDA
X[80:160, 7] = np.random.normal(3.5, 1, 80)   # High LF/HF

# Amusement samples (2)
y[160:] = 2
X[160:, 0] = np.random.normal(85, 12, 40)
X[160:, 2] = np.random.normal(35, 10, 40)
X[160:, 9] = np.random.normal(5, 1.5, 40)

# Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df['label'] = y
df['label_name'] = df['label'].map({0: 'Baseline', 1: 'Stress', 2: 'Amusement'})

print(f"Dataset shape: {X.shape}")
print(f"Class distribution:\n{pd.Series(y).value_counts().sort_index()}")

In [ ]:
# Train a Random Forest classifier for demonstration
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate
accuracy = model.score(X_test_scaled, y_test)
print(f"Model Accuracy: {accuracy:.4f}")

## 3. SHAP Explanations

SHAP (SHapley Additive exPlanations) uses game theory to compute feature importance.

In [ ]:
# Initialize SHAP explainer
shap_explainer = SHAPExplainer(
    model=model,
    model_type='tree',
    background_data=X_train_scaled[:50]
)

print("SHAP Explainer initialized")

In [ ]:
# Compute SHAP values for test set
shap_values = shap_explainer.compute_shap_values(
    X_test_scaled,
    feature_names=feature_names
)

print(f"SHAP values shape: {shap_values['shap_values'].shape}")
print(f"Base value: {shap_values['base_value']:.4f}")

In [ ]:
# Global feature importance (summary plot)
fig = shap_explainer.plot_summary(
    shap_values['shap_values'],
    feature_names,
    max_display=15,
    plot_type='dot'
)
plt.title('SHAP Feature Importance (All Classes)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Local explanation (waterfall plot for a single instance)
sample_idx = 0

fig = shap_explainer.plot_waterfall(
    shap_values['shap_values'],
    idx=sample_idx,
    feature_names=feature_names,
    max_display=10
)
plt.title(f'SHAP Waterfall Plot (Sample {sample_idx})', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Feature dependence plot
fig = shap_explainer.plot_dependence(
    shap_values['shap_values'],
    feature='hrv_rmssd',
    feature_names=feature_names,
    X=X_test_scaled,
    interaction_feature='auto'
)
plt.title('SHAP Dependence: HRV RMSSD', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Get top features
top_features = shap_explainer.get_top_features(
    shap_values['shap_values'],
    feature_names,
    n=10
)
print("Top 10 Most Important Features (SHAP):")
print(top_features.to_string(index=False))

## 4. LIME Explanations

LIME provides local interpretable explanations by fitting a simple model around the prediction.

In [ ]:
# Initialize LIME explainer
lime_explainer = LIMEExplainer(
    model=model,
    feature_names=feature_names,
    class_names=['Baseline', 'Stress', 'Amusement'],
    training_data=X_train_scaled
)

print("LIME Explainer initialized")

In [ ]:
# Explain a single instance
sample_idx = 0
lime_explanation = lime_explainer.explain_instance(
    X_test_scaled[sample_idx],
    num_features=10,
    num_samples=5000
)

print(f"\nPredicted Class: {lime_explanation['predicted_class_name']}")
print(f"Prediction Confidence: {lime_explanation['prediction_proba'].max():.4f}")
print(f"Local Model R² Score: {lime_explanation['score']:.4f}")

In [ ]:
# Plot LIME explanation
fig = lime_explainer.plot_explanation(
    lime_explanation,
    idx=sample_idx,
    show_predicted_proba=True
)
plt.show()

In [ ]:
# Compare LIME with SHAP
# Get SHAP importance for the same sample
shap_importance = {
    feature_names[i]: float(shap_values['shap_values'][sample_idx, i])
    for i in range(len(feature_names))
}

# Get LIME importance
lime_importance = {}
for feat, weight in lime_explanation['feature_weights'].items():
    if isinstance(feat, int):
        lime_importance[feature_names[feat]] = weight

# Compare
comparison = lime_explainer.compare_with_shap(
    lime_importance,
    shap_importance,
    top_n=10
)
print("\nLIME vs SHAP Comparison:")
print(comparison.to_string(index=False))

In [ ]:
# Plot comparison
fig = lime_explainer.plot_comparison(comparison)
plt.show()

## 5. Clinical Interpretation

The Clinical Interpreter provides physiologically-grounded explanations based on established standards (Task Force 1996).

In [ ]:
# Initialize Clinical Interpreter
clinical_interpreter = ClinicalInterpreter(
    class_names=['Baseline', 'Stress', 'Amusement']
)

# List supported physiological features
print("Supported Physiological Features:")
for feature in clinical_interpreter.list_supported_features()[:15]:
    info = clinical_interpreter.get_feature_info(feature)
    print(f"  {feature}: {info['description']} ({info['normal_range']})")

In [ ]:
# Create a sample with clinical features
# Use original (unscaled) values for clinical interpretation
clinical_sample = {
    'hr_mean': 95.0,       # Elevated heart rate
    'hrv_rmssd': 18.0,     # Low RMSSD (stress indicator)
    'hrv_sdnn': 35.0,      # Low SDNN
    'hrv_lf_hf_ratio': 3.8, # Elevated (sympathetic dominance)
    'eda_mean': 8.5,       # Elevated skin conductance
    'resp_rate': 22.0,     # Slightly elevated
    'temp_mean': 31.5,     # Slightly low (stress vasoconstriction)
}

# Assess individual features
print("\nIndividual Feature Assessment:")
print("=" * 60)

for feature, value in clinical_sample.items():
    finding = clinical_interpreter.assess_feature(feature, value)
    print(f"\n{feature}: {value}")
    print(f"  Deviation: {finding.deviation}")
    print(f"  Implication: {finding.stress_implication}")
    print(f"  Confidence: {finding.confidence:.2f}")

In [ ]:
# Full clinical interpretation
clinical_report = clinical_interpreter.interpret_prediction(
    prediction=1,  # Stress
    probabilities=[0.15, 0.75, 0.10],
    feature_values=clinical_sample,
    shap_values=shap_importance  # Include model explanations
)

print("\n" + "=" * 60)
print("CLINICAL INTERPRETATION REPORT")
print("=" * 60)
print(clinical_report['summary'])

In [ ]:
# Show recommendations
print("\nRecommendations:")
print("-" * 40)
for i, rec in enumerate(clinical_report['recommendations'], 1):
    print(f"{i}. {rec}")

print(f"\n{clinical_report['disclaimer']}")

In [ ]:
# Compare two sessions
session_baseline = {
    'hr_mean': 70.0,
    'hrv_rmssd': 45.0,
    'eda_mean': 3.0,
    'resp_rate': 14.0
}

session_stress = {
    'hr_mean': 95.0,
    'hrv_rmssd': 18.0,
    'eda_mean': 8.5,
    'resp_rate': 22.0
}

comparison = clinical_interpreter.compare_sessions(
    session_baseline,
    session_stress,
    session1_name='Baseline',
    session2_name='Stress'
)

print("\nSession Comparison:")
print(comparison.to_string(index=False))

## 6. Normal Range Visualization

In [ ]:
# Visualize feature values against normal ranges
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

features_to_plot = ['hr_mean', 'hrv_rmssd', 'hrv_lf_hf_ratio', 'eda_mean', 'resp_rate', 'temp_mean']

for idx, feature in enumerate(features_to_plot):
    ax = axes[idx]
    
    if feature in NORMAL_RANGES:
        normal = NORMAL_RANGES[feature]
        value = clinical_sample.get(feature, 0)
        
        # Create range bar
        range_min = normal.low * 0.5
        range_max = normal.high * 1.5
        
        # Plot normal range
        ax.axhspan(normal.low, normal.high, alpha=0.3, color='green', label='Normal Range')
        
        # Plot critical ranges if defined
        if normal.critical_low:
            ax.axhspan(range_min, normal.critical_low, alpha=0.3, color='red')
        if normal.critical_high:
            ax.axhspan(normal.critical_high, range_max, alpha=0.3, color='red')
        
        # Plot current value
        color = 'green' if normal.low <= value <= normal.high else 'red'
        ax.barh(0, value, color=color, height=0.3)
        ax.axvline(value, color=color, linestyle='--', linewidth=2)
        
        ax.set_xlim(range_min, range_max)
        ax.set_title(f'{feature}\n({normal.description})', fontsize=10)
        ax.set_xlabel(f'{normal.unit}')
        ax.set_yticks([])
        
        # Add value annotation
        ax.annotate(f'{value:.1f}', xy=(value, 0.15), ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Feature Values vs Normal Ranges', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Stress Level Distribution

In [ ]:
# Analyze stress levels across test samples
stress_assessments = []

for i in range(min(50, len(X_test))):
    # Create feature dict (use raw values)
    sample_features = {
        feature_names[j]: X_test[i, j]
        for j in range(len(feature_names))
    }
    
    pred = model.predict(X_test_scaled[i:i+1])[0]
    proba = model.predict_proba(X_test_scaled[i:i+1])[0]
    
    report = clinical_interpreter.interpret_prediction(
        prediction=int(pred),
        probabilities=proba.tolist(),
        feature_values=sample_features
    )
    
    stress_assessments.append({
        'sample': i,
        'prediction': int(pred),
        'stress_level': report['stress_assessment']['stress_level'],
        'stress_score': report['stress_assessment']['stress_score']
    })

assessment_df = pd.DataFrame(stress_assessments)

# Plot stress level distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stress level counts
level_counts = assessment_df['stress_level'].value_counts()
colors = {'low': 'green', 'normal': 'lightgreen', 'elevated': 'yellow', 'high': 'orange', 'very_high': 'red'}
axes[0].bar(level_counts.index, level_counts.values, color=[colors.get(l, 'gray') for l in level_counts.index])
axes[0].set_xlabel('Stress Level')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Stress Levels')

# Stress score distribution
axes[1].hist(assessment_df['stress_score'], bins=20, edgecolor='black', alpha=0.7)
axes[1].axvline(assessment_df['stress_score'].mean(), color='red', linestyle='--', label=f"Mean: {assessment_df['stress_score'].mean():.2f}")
axes[1].set_xlabel('Stress Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Stress Scores')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Generate Clinical Report (PDF-Ready)

In [ ]:
# Generate comprehensive report for a sample
sample_idx = 0

print("="*80)
print("COMPREHENSIVE STRESS DETECTION REPORT")
print("="*80)
print(f"\nReport Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Sample ID: {sample_idx}")

# Model Prediction
print("\n" + "-"*40)
print("MODEL PREDICTION")
print("-"*40)
pred = model.predict(X_test_scaled[sample_idx:sample_idx+1])[0]
proba = model.predict_proba(X_test_scaled[sample_idx:sample_idx+1])[0]
class_names = ['Baseline', 'Stress', 'Amusement']
print(f"Predicted Class: {class_names[pred]}")
print(f"Confidence: {max(proba)*100:.1f}%")
print("\nClass Probabilities:")
for name, p in zip(class_names, proba):
    bar = '█' * int(p * 30)
    print(f"  {name:12s}: {p*100:5.1f}% {bar}")

# SHAP Analysis
print("\n" + "-"*40)
print("FEATURE IMPORTANCE (SHAP)")
print("-"*40)
top_shap = shap_explainer.get_top_features(
    shap_values['shap_values'][sample_idx:sample_idx+1],
    feature_names,
    n=5
)
for _, row in top_shap.iterrows():
    direction = "+" if row['mean_shap'] > 0 else "-"
    print(f"  {row['feature']:20s}: {direction}{abs(row['mean_abs_shap']):.4f}")

# Clinical Assessment
print("\n" + "-"*40)
print("CLINICAL ASSESSMENT")
print("-"*40)
print(clinical_report['summary'])

print("\n" + "="*80)

## 9. Summary

### Key Findings

1. **SHAP Analysis**: Provides consistent, theoretically-grounded feature importance scores
2. **LIME Analysis**: Offers intuitive local explanations that are easy to communicate
3. **Clinical Interpretation**: Bridges the gap between ML predictions and clinical understanding

### Best Practices

- Always validate model predictions against clinical knowledge
- Use multiple explanation methods to ensure robustness
- Consider normal ranges when interpreting physiological features
- Include appropriate disclaimers for clinical applications

### Next Steps

1. Integrate explainability with real-time API
2. Add attention visualization for deep learning models
3. Develop interactive dashboard for clinical use

In [ ]:
print("\nWeek 7 Notebook Complete!")
print("\nModules demonstrated:")
print("  - SHAPExplainer: Global and local feature importance")
print("  - LIMEExplainer: Local interpretable explanations")
print("  - ClinicalInterpreter: Physiologically-grounded insights")
print("  - Normal ranges from Task Force (1996) standards")